In [1]:
# For tips on running notebooks in Google Colab, see
# https://docs.pytorch.org/tutorials/beginner/colab
%matplotlib inline

`torch.compile` End-to-End Tutorial =================================
**Author:** William Wen


`torch.compile` is the new way to speed up your PyTorch code!
`torch.compile` makes PyTorch code run faster by JIT-compiling PyTorch
code into optimized kernels, while requiring minimal code changes.

This tutorial covers an end-to-end example of training and evaluating a
real model with `torch.compile`. For a gentle introduction to
`torch.compile`, please check out [the introduction to torch.compile
tutorial](https://pytorch.org/tutorials/intermediate/torch_compile_tutorial.html).

**Required pip Dependencies**

-   `torch >= 2.0`
-   `torchvision`

<div style="width: 45%; float: left; padding: 20px;"><h2> What you will learn</h2><ul><li>How to apply <code>torch.compile</code> to a real model</li><li><code>torch.compile</code> speedups on a real model</li><li><code>torch.compile</code>'s first few iterations are expected to be slower due to compilation overhead</li></ul></div><div style="width: 45%; float: right; padding: 20px;"><h2> Prerequisites</h2><ul><li><a href="https://pytorch.org/tutorials/intermediate/torch_compile_tutorial.html">Introduction to torch.compile</a></li></ul></div>



In [2]:
# NOTE: a modern NVIDIA GPU (H100, A100, or V100) is recommended for this tutorial in
# order to reproduce the speedup numbers shown below and documented elsewhere.

import torch
import warnings

gpu_ok = False
if torch.cuda.is_available():
    device_cap = torch.cuda.get_device_capability()
    if device_cap in ((7, 0), (8, 0), (9, 0)):
        gpu_ok = True

if not gpu_ok:
    warnings.warn(
        "GPU is not NVIDIA V100, A100, or H100. Speedup numbers may be lower "
        "than expected."
    )

Let\'s demonstrate how using `torch.compile` can speed up a real model.
We will compare standard eager mode and `torch.compile` by evaluating
and training a `torchvision` model on random data.

Before we start, we need to define some utility functions.


In [3]:
# Returns the result of running `fn()` and the time it took for `fn()` to run,
# in seconds. We use CUDA events and synchronization for the most accurate
# measurements.
def timed(fn):
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    start.record()
    result = fn()
    end.record()
    torch.cuda.synchronize()
    return result, start.elapsed_time(end) / 1000


# Generates random input and targets data for the model, where `b` is
# batch size.
def generate_data(b):
    return (
        torch.randn(b, 3, 128, 128).cuda(),
        torch.randint(1000, (b,)).cuda(),
    )


N_ITERS = 10

from torchvision.models import densenet121


def init_model():
    return densenet121().cuda()

First, let\'s compare inference.

Note that in the call to `torch.compile`, we have the additional `mode`
argument, which we will discuss below.


In [4]:
model = init_model()

# Note that we generally recommend directly compiling a torch.nn.Module by calling
# its .compile() method.
model_opt = init_model()
model_opt.compile(mode="reduce-overhead")

inp = generate_data(16)[0]
with torch.no_grad():
    print("eager:", timed(lambda: model(inp))[1])
    print("compile:", timed(lambda: model_opt(inp))[1])

eager: 0.9869014892578125


/usr/local/lib/python3.12/dist-packages/torch/_inductor/compile_fx.py:321: UserWarning: TensorFloat32 tensor cores for float32 matrix multiplication available but not enabled. Consider setting `torch.set_float32_matmul_precision('high')` for better performance.
  warnings.warn(


compile: 78.0518046875


### 为什么会出现这个警告？

这个警告源于 `torch.compile` 的后端驱动 `TorchInductor`。在编译过程中，它会检查硬件能力。如果发现硬件支持 **TensorFloat32 (TF32)**（Ampere 架构及以后，如 A100, H100），但用户没有显式授权使用它，它就会发出此警告。

**如何查看和设置：**
PyTorch 提供了 `torch.set_float32_matmul_precision` 来控制这一行为：
- `'highest'`: 默认值，使用完整的 Float32，不使用 TF32。
- `'high'`: 允许在支持的硬件上使用 TF32，通常能显著提升性能且精度损失极小。
- `'medium'`: 允许使用 bfloat16 等更激进的加速。

In [5]:
import torch

# 1. 查看当前的矩阵乘法精度设置
current_setting = torch.get_float32_matmul_precision()
print(f"当前精度设置: {current_setting}")

# 2. 检查硬件是否支持 TF32
if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability()
    if major >= 8:
        print("您的硬件支持 TF32 (Ampere 架构或更高)。")
    else:
        print("您的硬件不支持 TF32。")

# 3. 如果你想消除警告并获得性能提升，可以执行以下代码：
# torch.set_float32_matmul_precision('high')
# print("已将精度设置为 'high' 以启用 TF32 Tensor Cores。")

当前精度设置: highest
您的硬件支持 TF32 (Ampere 架构或更高)。


Notice that `torch.compile` takes a lot longer to complete compared to
eager. This is because `torch.compile` compiles the model into optimized
kernels as it executes. In our example, the structure of the model
doesn\'t change, and so recompilation is not needed. So if we run our
optimized model several more times, we should see a significant
improvement compared to eager.


In [6]:
eager_times = []
for i in range(N_ITERS):
    inp = generate_data(16)[0]
    with torch.no_grad():
        _, eager_time = timed(lambda: model(inp))
    eager_times.append(eager_time)
    print(f"eager eval time {i}: {eager_time}")

print("~" * 10)

compile_times = []
for i in range(N_ITERS):
    inp = generate_data(16)[0]
    with torch.no_grad():
        _, compile_time = timed(lambda: model_opt(inp))
    compile_times.append(compile_time)
    print(f"compile eval time {i}: {compile_time}")
print("~" * 10)

import numpy as np

eager_med = np.median(eager_times)
compile_med = np.median(compile_times)
speedup = eager_med / compile_med
assert speedup > 1
print(
    f"(eval) eager median: {eager_med}, compile median: {compile_med}, speedup: {speedup}x"
)
print("~" * 10)

eager eval time 0: 0.02505625534057617
eager eval time 1: 0.02127872085571289
eager eval time 2: 0.020913152694702147
eager eval time 3: 0.020685823440551757
eager eval time 4: 0.02125619125366211
eager eval time 5: 0.021601280212402343
eager eval time 6: 0.020978687286376953
eager eval time 7: 0.020928512573242186
eager eval time 8: 0.022133760452270508
eager eval time 9: 0.021285888671875
~~~~~~~~~~
compile eval time 0: 0.09095475006103515
compile eval time 1: 0.006738944053649902
compile eval time 2: 0.008152064323425292
compile eval time 3: 0.00643891191482544
compile eval time 4: 0.006397952079772949
compile eval time 5: 0.006346752166748047
compile eval time 6: 0.006354944229125976
compile eval time 7: 0.006412288188934326
compile eval time 8: 0.00638156795501709
compile eval time 9: 0.006347775936126709
~~~~~~~~~~
(eval) eager median: 0.0212674560546875, compile median: 0.006405120134353638, speedup: 3.320383631935371x
~~~~~~~~~~


And indeed, we can see that running our model with `torch.compile`
results in a significant speedup. Speedup mainly comes from reducing
Python overhead and GPU read/writes, and so the observed speedup may
vary on factors such as model architecture and batch size. For example,
if a model\'s architecture is simple and the amount of data is large,
then the bottleneck would be GPU compute and the observed speedup may be
less significant.


You may also see different speedup results depending on the chosen
`mode` argument. The `"reduce-overhead"` mode uses CUDA graphs to
further reduce the overhead of Python. For your own models, you may need
to experiment with different modes to maximize speedup. You can read
more about modes
[here](https://pytorch.org/get-started/pytorch-2.0/#user-experience).

### 什么是 CUDA Graph？

在传统的 PyTorch 执行模式中，CPU 会逐个算子地启动 GPU 内核。对于高性能 GPU 来说，启动内核的时间（Overhead）有时甚至比计算本身还要长。

**CUDA Graph 的加速原理：**
1. **录制阶段 (Capture)**：在第一次运行时，记录下所有的内核启动顺序、参数和内存依赖关系。
2. **重放阶段 (Replay)**：后续运行时，不再逐个启动内核，而是将整个“图”一次性交给 GPU。这消除了 CPU 端的调度开销。

`torch.compile(mode="reduce-overhead")` 内部正是利用了这一技术来进一步压榨性能。


 modes such as reduce-overhead reduce the framework overhead by a lot more, but cost a small amount of extra memory. max-autotune compiles for a long time, trying to give you the fastest code it can generate.


这里提到的“额外的 memory”主要指的是 GPU 显存 (VRAM)。

具体原因如下：

1. CUDA Graph 存储：在 `reduce-overhead` 模式下，系统会录制 GPU 的操作序列（图）。为了保证性能，CUDA Graph 通常会预分配并锁定一块显存（称为 Mempool），用于存放图中所有算子的输入输出张量，这部分显存即使在不计算时也不会被释放。

2. Workspace 空间：`torch.compile` 的后端（如 Triton 或 cuBLAS）为了找到最快的算子实现，可能会分配一些临时的显存工作区（Workspace）来执行矩阵乘法或卷积的优化算法。

3. 内存 (RAM) 影响：虽然编译过程本身（Python 端的图分析和 TorchScript 转换）会消耗少量系统内存，但文档中强调的“成本”通常是指有限的显存资源，这在多模型运行或大 batch size 场景下需要特别注意。

In [7]:
# 演示如何手动查看 CUDA Graph 的简单概念（仅作说明）
# 在 torch.compile 中，这一切都是自动完成的。

print(f"当前模型是否使用了 CUDA Graph 优化 (reduce-overhead): {'True' if 'reduce-overhead' in str(model_opt) else '在编译配置中已启用'}")

# 注意：CUDA Graph 要求输入 Tensor 的形状是固定的。
# 如果形状经常变化，CUDA Graph 反而会因为不断重新录制而变慢。

当前模型是否使用了 CUDA Graph 优化 (reduce-overhead): 在编译配置中已启用


You may might also notice that the second time we run our model with
`torch.compile` is significantly slower than the other runs, although it
is much faster than the first run. This is because the
`"reduce-overhead"` mode runs a few warm-up iterations for CUDA graphs.

Now, let\'s consider comparing training.

In [8]:
model = init_model()
opt = torch.optim.Adam(model.parameters())


def train(mod, data):
    opt.zero_grad(True)
    pred = mod(data[0])
    loss = torch.nn.CrossEntropyLoss()(pred, data[1])
    loss.backward()
    opt.step()


eager_times = []
for i in range(N_ITERS):
    inp = generate_data(16)
    _, eager_time = timed(lambda: train(model, inp))
    eager_times.append(eager_time)
    print(f"eager train time {i}: {eager_time}")
print("~" * 10)

model = init_model()
opt = torch.optim.Adam(model.parameters())

# Note that because we are compiling a regular Python function, we do not
# call any .compile() method.
train_opt = torch.compile(train, mode="reduce-overhead")

compile_times = []
for i in range(N_ITERS):
    inp = generate_data(16)
    _, compile_time = timed(lambda: train_opt(model, inp))
    compile_times.append(compile_time)
    print(f"compile train time {i}: {compile_time}")
print("~" * 10)

eager_med = np.median(eager_times)
compile_med = np.median(compile_times)
speedup = eager_med / compile_med
assert speedup > 1
print(
    f"(train) eager median: {eager_med}, compile median: {compile_med}, speedup: {speedup}x"
)
print("~" * 10)

eager train time 0: 0.5910896606445313
eager train time 1: 0.0632176628112793
eager train time 2: 0.06072012710571289
eager train time 3: 0.05989683151245117
eager train time 4: 0.05914419174194336
eager train time 5: 0.05858303833007812
eager train time 6: 0.05961011123657227
eager train time 7: 0.06006988906860351
eager train time 8: 0.05988249588012695
eager train time 9: 0.05916262435913086
~~~~~~~~~~


W0502 02:43:00.706000 9628 torch/_logging/_internal.py:1204] [4/0] Profiler function <class 'torch.autograd.profiler.record_function'> will be ignored


compile train time 0: 192.576796875
compile train time 1: 2.465833984375
compile train time 2: 0.01946316719055176
compile train time 3: 0.015426560401916504
compile train time 4: 0.014180352210998535
compile train time 5: 0.014101504325866699
compile train time 6: 0.013985792160034179
compile train time 7: 0.013911040306091308
compile train time 8: 0.01387007999420166
compile train time 9: 0.013853695869445801
~~~~~~~~~~
(train) eager median: 0.059889663696289064, compile median: 0.014140928268432617, speedup: 4.235200303645076x
~~~~~~~~~~


### 关于 `torch.compile` 的执行时机

当你执行 `train_opt = torch.compile(train)` 时，实际上**并没有**立即发生代码优化。

*   **延迟编译 (Lazy Compilation)**：真正的编译（包括图捕获、算子融合等）是发生在**第一次调用** `train_opt` 并且有具体形状和数据类型的张量输入时。这也是为什么第一个 iteration 总是显著慢于后续运行的原因。
*   **返回对象**：`torch.compile` 返回的是一个原函数的包装器（Optimized Function Holder）。它内部维护了一个缓存，如果后续输入的形状和类型与之前编译过的一致，它就会直接跳过编译，运行优化后的二进制内核。

### 为什么编译整个训练步骤（Train Step）性能更好？

虽然我们可以只编译模型 `model.compile()`，但编译整个训练函数通常能提供更显著的加速，原因包括：

1.  **跨算子融合 (Fusion)**：编译器可以将前向传播、损失函数计算和反向传播中的算子融合在一起。例如，它可以减少数据在显存和寄存器之间的读写次数。
2.  **减少 Python 框架开销**：训练循环中包含大量的 Python 指令。编译整个函数可以将这些逻辑转化为高效的 Triton 或 CUDA 内核，最大程度地减少 Python 解释器的参与。
3.  **全局图优化**：编译器能够对整个训练步长进行全局分析，而不仅仅局限于模型内部，从而发现更多并行化和优化的机会。

Again, we can see that `torch.compile` takes longer in the first
iteration, as it must compile the model, but in subsequent iterations,
we see significant speedups compared to eager.

We remark that the speedup numbers presented in this tutorial are for
demonstration purposes only. Official speedup values can be seen at the
[TorchInductor performance
dashboard](https://hud.pytorch.org/benchmark/compilers).


### 深入验证：查看 `torch.compile` 的融合行为

我们可以利用 `torch._inductor.config.debug = True` 或者查看生成的 Triton 代码来验证上述理论。下面的代码演示了如何观察编译器如何处理一个简单的“线性层 + 激活函数 + 损失函数”组合。

In [11]:
import torch
import torch._logging

# 重置并设置日志级别以打印生成的代码
torch._dynamo.reset()
torch._logging.set_logs(output_code=True)

def simple_train_step(x, target, model, opt):
    opt.zero_grad(True)
    pred = model(x)
    # 这里的 ReLU 和 MSELoss 很容易被融合
    loss = torch.nn.functional.mse_loss(torch.relu(pred), target)
    loss.backward()
    opt.step()
    return loss

# 编译整个函数
compiled_train = torch.compile(simple_train_step)

# 准备测试数据
x = torch.randn(16, 10).cuda()
target = torch.randn(16, 10).cuda()
model = torch.nn.Linear(10, 10).cuda()
opt = torch.optim.SGD(model.parameters(), lr=0.01)

print("执行编译后的训练步骤 (Triton 代码将显示在下方)...")
# 执行会触发编译并打印 Triton 代码
loss = compiled_train(x, target, model, opt)

V0502 02:48:59.271000 9628 torch/_inductor/codecache.py:1250] [1/0_1] [__output_code] Output code: 
V0502 02:48:59.271000 9628 torch/_inductor/codecache.py:1250] [1/0_1] [__output_code] # AOT ID: ['5_forward']
V0502 02:48:59.271000 9628 torch/_inductor/codecache.py:1250] [1/0_1] [__output_code] from ctypes import c_void_p, c_long, c_int
V0502 02:48:59.271000 9628 torch/_inductor/codecache.py:1250] [1/0_1] [__output_code] import torch
V0502 02:48:59.271000 9628 torch/_inductor/codecache.py:1250] [1/0_1] [__output_code] import math
V0502 02:48:59.271000 9628 torch/_inductor/codecache.py:1250] [1/0_1] [__output_code] import random
V0502 02:48:59.271000 9628 torch/_inductor/codecache.py:1250] [1/0_1] [__output_code] import os
V0502 02:48:59.271000 9628 torch/_inductor/codecache.py:1250] [1/0_1] [__output_code] import tempfile
V0502 02:48:59.271000 9628 torch/_inductor/codecache.py:1250] [1/0_1] [__output_code] from math import inf, nan
V0502 02:48:59.271000 9628 torch/_inductor/codecache.p

执行编译后的训练步骤 (Triton 代码将显示在下方)...


V0502 02:48:59.453000 9628 torch/_inductor/codecache.py:1250] [5/0] [__output_code] Output code: 
V0502 02:48:59.453000 9628 torch/_inductor/codecache.py:1250] [5/0] [__output_code] # AOT ID: ['6_inference']
V0502 02:48:59.453000 9628 torch/_inductor/codecache.py:1250] [5/0] [__output_code] from ctypes import c_void_p, c_long, c_int
V0502 02:48:59.453000 9628 torch/_inductor/codecache.py:1250] [5/0] [__output_code] import torch
V0502 02:48:59.453000 9628 torch/_inductor/codecache.py:1250] [5/0] [__output_code] import math
V0502 02:48:59.453000 9628 torch/_inductor/codecache.py:1250] [5/0] [__output_code] import random
V0502 02:48:59.453000 9628 torch/_inductor/codecache.py:1250] [5/0] [__output_code] import os
V0502 02:48:59.453000 9628 torch/_inductor/codecache.py:1250] [5/0] [__output_code] import tempfile
V0502 02:48:59.453000 9628 torch/_inductor/codecache.py:1250] [5/0] [__output_code] from math import inf, nan
V0502 02:48:59.453000 9628 torch/_inductor/codecache.py:1250] [5/0] [_

**如何阅读这些输出以及观察点：**

1. **AOT ID (Forward/Backward)**：输出中会区分 `*_forward` 和 `*_backward`，分别代表前向和反向传播的优化代码。

2. **算子融合 (Fusion)**：请在输出中搜索 `def triton_` 开头的函数。你会发现原本在 Python 中独立的 relu 和 mse_loss 计算，现在被写在了一个单一的 Triton kernel 中。例如，你会看到类似 `tl.where(x > 0, x, 0)` (ReLU) 和平方差计算在同一个循环里，而不是两个独立的内核调用。这正是优化 Train Loop 而不是 Model 本身带来的额外收益。

3. **生成的内核路径**：日志中还提到了 `Output code written to: /tmp/...`，那是完整生成的优化代码文件路径。

4. **反向传播优化**：`loss.backward()` 产生的梯度计算会直接与 `opt.step()` 的权重更新逻辑在内存层面进行优化（如果使用了特定优化器）。 比如使用 SGD 等，在融合后的代码中，你会看到生成的 Triton 代码直接 Load `param`，在寄存器里加上梯度项，然后直接 Store 回 `param`。整个过程 param.grad 可能根本没有发生实际的显存写入。这就是“内存层面”的优化。



**最佳实践**： PyTorch 官方的推荐路径也是渐进式的：

- 先只编译核心网络：model = torch.compile(model)
- 确保没有问题，且你的训练步逻辑相对清晰纯粹时，再尝试编译整个 train_step。
- 如果引入 train_step 编译导致性能下降或报错，果断退回到只编译 model。

Conclusion
==========

In this tutorial, we applied `torch.compile` to training and inference
on a real model, demonstrating speedups.

Importantly, we note that the first few iterations of a compiled model
are slower than eager mode due to compilation overhead, but subsequent
iterations are expected to have speedups.

For a gentle introduction to `torch.compile`, please check out [the
introduction to torch.compile
tutorial](https://pytorch.org/tutorials/intermediate/torch_compile_tutorial.html).

To troubleshoot issues and to gain a deeper understanding of how to
apply `torch.compile` to your code, check out [the torch.compile
programming
model](https://docs.pytorch.org/docs/stable/user_guide/torch_compiler/compile/programming_model.html).

We hope that you will give `torch.compile` a try!
